# QSP HPC Tools — Feature Walkthrough

This notebook demonstrates the core capabilities of `qsp-hpc-tools`:

1. **Configuration & setup** — CLI-based credential management
2. **QSPSimulator** — the main callable interface for generating simulations
3. **Three-tier caching** — local pool, HPC test statistics, HPC full simulations
4. **Content-based cache invalidation** — automatic re-computation when inputs change
5. **Multi-scenario support** — shared virtual patients across therapy protocols
6. **SimulationPoolManager** — direct pool inspection and management
7. **`simulate_with_parameters`** — running specific parameter sets (e.g., posterior predictive checks)

**Prerequisites:**
- `pip install qsp-hpc-tools`
- For HPC features: credentials configured via `qsp-hpc setup`

## 1. Configuration & setup

`qsp-hpc-tools` uses a single global config file at `~/.config/qsp-hpc/credentials.yaml`. The CLI provides an interactive setup wizard that detects SSH hosts, tests connections, and optionally creates a Python virtual environment on the cluster.

```bash
$ qsp-hpc setup    # interactive wizard
$ qsp-hpc test     # verify HPC connection and SLURM access
$ qsp-hpc info     # display current configuration
$ qsp-hpc logs     # view job logs
```

In [ ]:
# You can also check configuration programmatically
from qsp_hpc.batch.hpc_job_manager import HPCJobManager

try:
    manager = HPCJobManager()
    print(f"HPC host: {manager.config.ssh_host}")
    print(f"Remote base: {manager.config.remote_base_dir}")
    print(f"Pool path: {manager.config.simulation_pool_path}")
except FileNotFoundError:
    print("No credentials configured — run 'qsp-hpc setup' first")

## 2. Inputs: priors and calibration targets

`QSPSimulator` takes two inputs:
- **Priors CSV** — parameter names and lognormal prior distributions
- **Calibration targets** — a directory of YAML files defining observables to extract from simulation timecourses

Calibration target YAMLs are compatible with [MAPLE](https://github.com/popellab/maple), a framework for LLM-assisted extraction of calibration data from the literature. Each YAML specifies:
- `observable.species` — which model species are needed
- `observable.code` — a Python function to compute the observable from timecourse data
- `empirical_data` — observed values with uncertainty for calibration

In [ ]:
import pandas as pd
import yaml
from pathlib import Path

# Inspect the priors
priors = pd.read_csv("data/priors.csv")
priors

In [ ]:
# Inspect a calibration target YAML
target_path = "data/calibration_targets/control/tumor_volume_day60.yaml"
target = yaml.safe_load(open(target_path))
print(yaml.dump(target, default_flow_style=False, sort_keys=False))

## 3. QSPSimulator — the callable interface

`QSPSimulator` is the main entry point. It wraps a MATLAB QSP model as a Python callable that returns `(parameters, observables)` pairs. Behind the scenes it manages caching, HPC job submission, and result collection.

Calling `sim(n)` triggers a three-tier cache lookup:

1. **Local pool** — previously downloaded results on disk
2. **HPC test statistics** — pre-computed summary statistics on the cluster (avoids downloading full timecourses)
3. **HPC full simulations** — complete simulation outputs on the cluster

Only when all tiers miss does it submit new SLURM array jobs. This means re-running the same call is essentially free after the first execution.

In [ ]:
from qsp_hpc import QSPSimulator

sim = QSPSimulator(
    priors_csv="data/priors.csv",
    calibration_targets="data/calibration_targets/control",
    model_script="tumor_immune_model",       # name of your MATLAB model
    model_version="v1",
    scenario="control",
    seed=42,
    max_tasks=20,                            # SLURM array parallelism
)

# Request 1000 simulations — triggers cache lookup, then HPC if needed
theta, x = sim(1000)
print(f"Parameters: {theta.shape}  (n_samples x n_params)")
print(f"Observables: {x.shape}  (n_samples x n_targets)")

In [ ]:
# Calling again returns cached results instantly — no HPC submission
theta2, x2 = sim(1000)
import numpy as np
assert np.array_equal(theta, theta2), "Cache miss — should not happen"
print("Cache hit: identical results returned")

## 4. Content-based cache invalidation

Pool directories are keyed by a hash of the semantically meaningful inputs: priors, calibration targets (observable code, units, species), model script, model version, and scenario. Changing any of these automatically creates a new pool — no manual cache management needed.

Cosmetic changes (renaming a description, reordering rows) do **not** invalidate the cache.

In [ ]:
# The simulator's pool directory encodes the config hash
print(f"Pool directory: {sim.pool.pool_dir}")
print(f"Config hash:    {sim.pool.config_hash}")

# Creating a simulator with a different model_version → different pool
sim_v2 = QSPSimulator(
    priors_csv="data/priors.csv",
    calibration_targets="data/calibration_targets/control",
    model_script="tumor_immune_model",
    model_version="v2",                      # changed!
    scenario="control",
    seed=42,
)
print(f"\nv2 pool directory: {sim_v2.pool.pool_dir}")
print(f"v2 config hash:    {sim_v2.pool.config_hash}")
print("Different hash → different pool → no stale cache")

## 5. Multi-scenario support

QSP workflows often evaluate the same virtual patients under different therapy protocols. `QSPSimulator` supports this natively:

- `cache_sampling_seed` ensures identical parameter draws across scenarios
- Each scenario gets its own independent simulation pool
- Downstream analysis can perform joint NaN filtering to retain only patients valid under all conditions

In [ ]:
SCENARIOS = ["control", "treatment"]
simulators = {}
results = {}

for scenario in SCENARIOS:
    sim_scen = QSPSimulator(
        priors_csv="data/priors.csv",
        calibration_targets=f"data/calibration_targets/{scenario}",
        model_script="tumor_immune_model",
        model_version="v1",
        scenario=scenario,
        seed=42,
        cache_sampling_seed=2025,            # ensures identical theta across scenarios
        max_tasks=20,
    )
    theta_s, x_s = sim_scen(1000)
    simulators[scenario] = sim_scen
    results[scenario] = {"theta": theta_s, "x": x_s}
    print(f"{scenario}: {theta_s.shape[0]} patients, {x_s.shape[1]} observables")

# Verify same parameters were drawn for both scenarios
assert np.array_equal(
    results["control"]["theta"], results["treatment"]["theta"]
), "Parameters should be identical across scenarios"
print("\nSame virtual patients used across both scenarios")

## 6. SimulationPoolManager — pool inspection

The `SimulationPoolManager` provides direct access to the local simulation cache. You can inspect pool contents, list scenarios, and enumerate batches — useful for debugging or monitoring long-running campaigns.

In [ ]:
from qsp_hpc.simulation.simulation_pool import SimulationPoolManager

# Access the pool through the simulator
pool = sim.pool
info = pool.get_pool_info()

print(f"Pool directory:  {info['pool_dir']}")
print(f"Model version:   {info['model_version']}")
print(f"Config hash:     {info['config_hash']}")
print(f"Total batches:   {info['n_batches']}")
print(f"Total sims:      {info['total_simulations']}")
print(f"Scenarios:       {info['scenarios']}")
print()
for scen, scen_info in info["scenario_info"].items():
    print(f"  {scen}: {scen_info['total_simulations']} sims in {scen_info['n_batches']} batch(es)")

In [ ]:
# List all pools in a cache directory
all_pools = SimulationPoolManager.list_pools(sim.pool.pool_dir.parent)
for p in all_pools:
    print(f"{p['pool_dir'].name}: {p['total_simulations']} sims, scenarios={p['scenarios']}")

## 7. Running specific parameter sets

`simulate_with_parameters` runs simulations for a given parameter matrix rather than sampling from the prior. This is used for posterior predictive checks — after inferring a posterior, you can re-simulate with posterior samples to validate the model.

Results are cached using a hash of the parameter matrix, so identical posterior samples produce cache hits.

In [ ]:
# Suppose we have posterior samples (here, just a subset of prior samples for demo)
posterior_samples = theta[:50]

# Re-simulate with those specific parameters
x_posterior = sim.simulate_with_parameters(posterior_samples)
print(f"Posterior predictive: {x_posterior.shape}  (n_posterior_samples x n_targets)")

# Calling again with the same parameters → cache hit (theta hash matches)
x_posterior_2 = sim.simulate_with_parameters(posterior_samples)
assert np.array_equal(x_posterior, x_posterior_2)
print("Cache hit on posterior predictive simulations")

## 8. HPC job management

When the cache misses, `HPCJobManager` handles the full lifecycle:

1. **Sync codebase** — rsync the MATLAB model to the cluster
2. **Upload inputs** — priors CSV and calibration targets
3. **Submit SLURM array job** — parallel simulation across `max_tasks` workers
4. **Monitor progress** — poll job status until completion
5. **Derive test statistics** — compute observables from raw timecourses on the cluster
6. **Download results** — transfer computed test statistics back to the local pool

All of this happens transparently inside `sim(n)`. The `HPCJobManager` can also be used directly for lower-level control:

In [ ]:
# Direct HPCJobManager usage (requires configured credentials)
from qsp_hpc.batch.hpc_job_manager import HPCJobManager

try:
    manager = HPCJobManager()

    # Check what's on the cluster
    hpc_pool_path = f"{manager.config.simulation_pool_path}/v1_abcd1234_control"
    has_sims = manager.check_hpc_simulations(hpc_pool_path, expected_n_sims=1000)
    print(f"HPC has simulations: {has_sims}")

    # Submit a job directly
    # manager.submit_simulation_job(
    #     priors_csv="data/priors.csv",
    #     model_script="tumor_immune_model",
    #     n_simulations=1000,
    #     n_tasks=20,
    #     seed=42,
    #     scenario="control",
    # )
except FileNotFoundError:
    print("HPCJobManager requires credentials — run 'qsp-hpc setup' first")
    print("When configured, it handles SSH, SLURM submission, and result collection")